# Sahayak — CAT Operator Assistant on Colab

Runs the whole thing on one port and gives you an HTTPS link you can open on your laptop or phone.

**The microphone will not work over plain `http://`.** Browsers only allow mic access on a secure
origin, which is why step 4 puts a Cloudflare tunnel in front. Open the `https://...trycloudflare.com`
link it prints, not the local address.

**Runtime → Change runtime type → T4 GPU** before you start. It works on CPU too, just slower
(roughly 3-5s per spoken question instead of well under a second).


## 1. Check the runtime


In [ ]:
import torch, sys
print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
print('cuda    ', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

# torch >= 2.6 is required: the Hindi Whisper and MuRIL checkpoints are
# pickle-only, and transformers refuses to load those on older torch.
major, minor = (int(x) for x in torch.__version__.split('.')[:2])
print('torch OK' if (major, minor) >= (2, 6) else 'TORCH TOO OLD - setup will upgrade it')


## 2. Clone and set everything up

Installs ffmpeg, the Python deps, builds the frontend and downloads ~3.5 GB of models.
Takes roughly 5-10 minutes the first time.


In [ ]:
!git clone -q https://github.com/krishagarwal314/sahayak-cat-operator-assistant.git /content/sahayak || echo 'already cloned'
%cd /content/sahayak
!bash scripts/setup_cloud.sh


## 3. Train the intent classifier (optional, ~3 minutes on a T4)

Skip this and the assistant still works — the router just stops at stage L2.
Train it if you want to show stage L3 answering questions in the demo.


In [ ]:
%cd /content/sahayak/backend
!python -m app.ai.intent.build_dataset
!python -m app.ai.intent.train --epochs 6

import json, pathlib
metrics = pathlib.Path('models/intent-classifier/metrics.json')
if metrics.exists():
    m = json.loads(metrics.read_text())
    print(f"validation accuracy: {m['best_val_accuracy']:.1%} over {m['labels']} labels")
    print('weakest labels:', m['weakest_labels'])


## 4. Start the server and open the tunnel

Wait for the `https://....trycloudflare.com` line, then open it in a new tab.
Sign in with **OP1001 / cat1234**.


In [ ]:
import os, re, subprocess, time, pathlib, urllib.request

ROOT = pathlib.Path('/content/sahayak')
CLOUDFLARED = pathlib.Path('/usr/local/bin/cloudflared')

# cloudflared gives us an HTTPS origin, which is what the microphone needs.
if not CLOUDFLARED.exists():
    print('downloading cloudflared ...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CLOUDFLARED,
    )
    CLOUDFLARED.chmod(0o755)

env = {**os.environ, 'EAGER_LOAD_MODELS': '1', 'PYTHONUNBUFFERED': '1'}
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=ROOT / 'backend', env=env,
    stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT,
)

# Models load at boot so the first question of the demo is not the slow one.
print('loading models, this takes a minute on first run ...')
for _ in range(240):
    try:
        urllib.request.urlopen('http://localhost:8000/api/system/health', timeout=2)
        print('server up')
        break
    except Exception:
        time.sleep(1)
else:
    print('server did not start - run the log cell below')

tunnel = subprocess.Popen(
    [str(CLOUDFLARED), 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
for line in tunnel.stdout:
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if found:
        public_url = found.group(0)
        break

print()
print('=' * 62)
print('  OPEN THIS:', public_url)
print('  sign in  : OP1001 / cat1234')
print('=' * 62)


## 5. Check the models actually loaded

Every role should say `loaded`. Anything that says `missing` will silently degrade —
the app keeps working, but that part of the demo falls back.


In [ ]:
import json, urllib.request
status = json.load(urllib.request.urlopen('http://localhost:8000/api/system/models'))
reg = status['registry']
print('profile:', reg['profile'], '| device:', reg['device'])
for role, name in reg['configured'].items():
    state = 'FAILED' if role in reg['failed'] else ('loaded' if role in reg['loaded'] else 'idle')
    print(f'  {role:11s} {state:8s} {name}')
for role, why in reg['failed'].items():
    print(f'  ! {role}: {why}')


## 6. Smoke-test the voice pipeline before you present

Synthesises a Hindi sentence and plays it back. If you hear it, TTS is working.


In [ ]:
import urllib.request, json
from IPython.display import Audio, display

token = json.load(urllib.request.urlopen(urllib.request.Request(
    'http://localhost:8000/api/auth/login',
    data=json.dumps({'username': 'OP1001', 'password': 'cat1234'}).encode(),
    headers={'Content-Type': 'application/json'})))['token']

ask = urllib.request.Request('http://localhost:8000/api/assistant/ask',
    data=json.dumps({'machine_id': 'EXC001', 'text': 'कितना ईंधन बचा है',
                     'language': 'hi', 'speak': True}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {token}'})
result = json.load(urllib.request.urlopen(ask))

print('intent :', result['route']['intent'], 'via', result['route']['stage'],
      f"({result['route']['total_ms']} ms)")
print('answer :', result['reply']['text']['hi'])

if result.get('audio'):
    import base64
    display(Audio(base64.b64decode(result['audio']['base64']), autoplay=False))
else:
    print('no audio - TTS model did not load, the browser will synthesise instead')


## Keeping it alive

Colab disconnects an idle notebook. Keep the tab open and interact with it every so often.
If it dies mid-demo you lose the models too — which is the main reason Lightning AI
is the better choice for a scheduled demo: its filesystem persists, so models download once.

Server logs, if something looks wrong:


In [ ]:
!tail -40 /content/server.log


## Stop everything


In [ ]:
server.terminate(); tunnel.terminate()
print('stopped')
